In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    silhouette_score,
    confusion_matrix,
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_style("whitegrid")

In [ ]:
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="diagnosis")  # 0 = malignant, 1 = benign

print("Shape:", X.shape)
print(y.value_counts().rename({0: "malignant", 1: "benign"}))
print("Nulls:", X.isnull().sum().sum())

In [ ]:
# Class distribution
fig, ax = plt.subplots(figsize=(5, 3))
y.map({0: "Malignant", 1: "Benign"}).value_counts().plot(kind="bar", ax=ax)
ax.set_title("Class Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(X.corr(), cmap="coolwarm", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Feature Correlation Matrix (30 features)")
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions by class
features_to_plot = ["mean radius", "mean texture", "mean concavity", "worst concave points"]
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for ax, feat in zip(axes, features_to_plot):
    for label, name in [(0, "Malignant"), (1, "Benign")]:
        ax.hist(X.loc[y == label, feat], bins=30, alpha=0.5, label=name)
    ax.set_title(feat)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 2D PCA projection, true labels
X_scaled_preview = StandardScaler().fit_transform(X)
X_pca_preview = PCA(n_components=2).fit_transform(X_scaled_preview)

plt.figure(figsize=(6, 5))
for label, name, color in [(0, "Malignant", "crimson"), (1, "Benign", "steelblue")]:
    mask = y == label
    plt.scatter(X_pca_preview[mask, 0], X_pca_preview[mask, 1],
                label=name, alpha=0.6, s=20, c=color)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("2D PCA Projection (True Labels)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
# Standardize
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# PCA, 95% variance
pca = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print(f"PCA reduced {X_train_scaled.shape[1]}D -> {X_train_pca.shape[1]}D")
print(f"Cumulative variance: {pca.explained_variance_ratio_.cumsum().round(3)}")

In [ ]:
# Scree plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[1].plot(range(1, len(pca.explained_variance_ratio_) + 1),
             pca.explained_variance_ratio_.cumsum(), marker='o')
axes[1].axhline(0.95, color='red', linestyle='--')
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Variance")
plt.tight_layout()
plt.show()

In [ ]:
# Eval helpers
def evaluate_gmm(model, X, y_true, model_name):
    y_pred = model.predict(X)
    log_lik = model.score(X)
    sil = silhouette_score(X, y_pred) if len(np.unique(y_pred)) > 1 else np.nan
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    print(f"{model_name}: log_lik={log_lik:.3f} sil={sil:.3f} ARI={ari:.3f} NMI={nmi:.3f}")
    return {'model': model_name, 'log_lik': log_lik, 'silhouette': sil,
            'ARI': ari, 'NMI': nmi, 'y_pred': y_pred}


def aligned_confusion(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    if cm[0, 0] + cm[1, 1] < cm[0, 1] + cm[1, 0]:
        y_pred = 1 - y_pred
        cm = confusion_matrix(y_true, y_pred)
    return cm, y_pred

In [ ]:
# PCA vs no-PCA comparison
km = KMeans(n_clusters=2, n_init=30, random_state=RANDOM_STATE)
km.fit(X_train_pca)
y_km_test = km.predict(X_test_pca)
cm_km = aligned_confusion(y_test.values, y_km_test)[0]
print(f"K-means, PCA (10D)         ARI={adjusted_rand_score(y_test, y_km_test):.3f} "
      f"Acc={(cm_km[0,0]+cm_km[1,1])/cm_km.sum():.3f}")

gmm_2pc = GaussianMixture(n_components=2, covariance_type='full',
                           n_init=30, random_state=RANDOM_STATE)
gmm_2pc.fit(X_train_pca[:, :2])
y_pred_2pc = gmm_2pc.predict(X_test_pca[:, :2])
cm_2pc = aligned_confusion(y_test.values, y_pred_2pc)[0]
print(f"GMM, 2 PCs                 ARI={adjusted_rand_score(y_test, y_pred_2pc):.3f} "
      f"Acc={(cm_2pc[0,0]+cm_2pc[1,1])/cm_2pc.sum():.3f}")

gmm_diag_pca = GaussianMixture(n_components=2, covariance_type='diag',
                                n_init=30, random_state=RANDOM_STATE)
gmm_diag_pca.fit(X_train_pca)
y_pred_diag_pca = gmm_diag_pca.predict(X_test_pca)
cm_diag_pca = aligned_confusion(y_test.values, y_pred_diag_pca)[0]
print(f"GMM diag cov, PCA (10D)     ARI={adjusted_rand_score(y_test, y_pred_diag_pca):.3f} "
      f"Acc={(cm_diag_pca[0,0]+cm_diag_pca[1,1])/cm_diag_pca.sum():.3f}")

gmm_diag_noPCA = GaussianMixture(n_components=2, covariance_type='diag',
                                  n_init=30, random_state=RANDOM_STATE)
gmm_diag_noPCA.fit(X_train_scaled)
y_pred_diag_noPCA = gmm_diag_noPCA.predict(X_test_scaled)
cm_diag_noPCA = aligned_confusion(y_test.values, y_pred_diag_noPCA)[0]
print(f"GMM diag cov, no PCA (30D)  ARI={adjusted_rand_score(y_test, y_pred_diag_noPCA):.3f} "
      f"Acc={(cm_diag_noPCA[0,0]+cm_diag_noPCA[1,1])/cm_diag_noPCA.sum():.3f}")

gmm_full_noPCA = GaussianMixture(n_components=2, covariance_type='full',
                                  n_init=30, random_state=RANDOM_STATE)
gmm_full_noPCA.fit(X_train_scaled)
y_pred_full_noPCA = gmm_full_noPCA.predict(X_test_scaled)
cm_full_noPCA = aligned_confusion(y_test.values, y_pred_full_noPCA)[0]
print(f"GMM full cov, no PCA (30D)  ARI={adjusted_rand_score(y_test, y_pred_full_noPCA):.3f} "
      f"Acc={(cm_full_noPCA[0,0]+cm_full_noPCA[1,1])/cm_full_noPCA.sum():.3f}")
labels = ["K-means\nPCA (10D)", "GMM\n2 PCs", "GMM diag\nPCA (10D)", "GMM diag\nno PCA (30D)", "GMM full\nno PCA (30D)"]
aris = [adjusted_rand_score(y_test, y_km_test), adjusted_rand_score(y_test, y_pred_2pc),
        adjusted_rand_score(y_test, y_pred_diag_pca), adjusted_rand_score(y_test, y_pred_diag_noPCA),
        adjusted_rand_score(y_test, y_pred_full_noPCA)]
colors = ['#888888', '#888888', '#d62728', '#888888', '#2ca02c']

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(labels, aris, color=colors, width=0.6)
for bar, val in zip(bars, aris):
    ax.annotate(f'{val:.2f}', (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Adjusted Rand Index (test)')
ax.set_title('Effect of PCA and Covariance Structure on Clustering Quality')
ax.set_ylim(0, max(aris) * 1.25)
plt.xticks(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Final model: full covariance GMM, no PCA
gmm_final = GaussianMixture(
    n_components=2,
    covariance_type='full',
    n_init=30,
    random_state=RANDOM_STATE,
)
gmm_final.fit(X_train_scaled)

y_pred_train = gmm_final.predict(X_train_scaled)
y_pred_test = gmm_final.predict(X_test_scaled)

cm_train = aligned_confusion(y_train.values, y_pred_train)[0]
cm_test, y_pred_test_aligned = aligned_confusion(y_test.values, y_pred_test)

acc_train = (cm_train[0,0]+cm_train[1,1])/cm_train.sum()
acc_test = (cm_test[0,0]+cm_test[1,1])/cm_test.sum()

print(f"Train accuracy: {acc_train:.3f}")
print(f"Test accuracy:  {acc_test:.3f}")
print(f"Train ARI: {adjusted_rand_score(y_train, y_pred_train):.3f}")
print(f"Test ARI:  {adjusted_rand_score(y_test, y_pred_test):.3f}")
print(f"Test NMI:  {normalized_mutual_info_score(y_test, y_pred_test):.3f}")
print(f"Sensitivity: {cm_test[0,0]/(cm_test[0,0]+cm_test[0,1]):.3f}")
print(f"Specificity: {cm_test[1,1]/(cm_test[1,0]+cm_test[1,1]):.3f}")

In [ ]:
# Confusion matrix plot
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Malignant', 'Benign'],
            yticklabels=['Malignant', 'Benign'], ax=ax)
ax.set_xlabel('Predicted Cluster')
ax.set_ylabel('True Diagnosis')
ax.set_title('Final Model -- Test Set Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster visualization (PCA used only for 2D plotting, not modeling)
X_test_2d = X_test_pca[:, :2]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for label, name, color in [(0, 'Malignant', 'crimson'), (1, 'Benign', 'steelblue')]:
    mask = y_test.values == label
    axes[0].scatter(X_test_2d[mask, 0], X_test_2d[mask, 1], c=color, label=name, alpha=0.6, s=30)
axes[0].set_title('True Labels')
axes[0].legend()

for label, name, color in [(0, 'Cluster M', 'crimson'), (1, 'Cluster B', 'steelblue')]:
    mask = y_pred_test_aligned == label
    axes[1].scatter(X_test_2d[mask, 0], X_test_2d[mask, 1], c=color, label=name, alpha=0.6, s=30)
axes[1].set_title('Final Model Predictions')
axes[1].legend()

misclassified_mask = y_test.values != y_pred_test_aligned
axes[2].scatter(X_test_2d[:, 0], X_test_2d[:, 1], c='lightgray', alpha=0.5, s=30)
axes[2].scatter(X_test_2d[misclassified_mask, 0], X_test_2d[misclassified_mask, 1],
                c='red', s=80, marker='x', linewidths=2, label='Misclassified')
axes[2].set_title(f'Misclassified ({misclassified_mask.sum()}/{len(y_test)})')
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Bayesian GMM confirmation (Dirichlet prior), no PCA
bgmm_final = BayesianGaussianMixture(
    n_components=2,
    covariance_type='full',
    weight_concentration_prior_type='dirichlet_distribution',
    weight_concentration_prior=1.0,
    n_init=30,
    max_iter=500,
    random_state=RANDOM_STATE,
)
bgmm_final.fit(X_train_scaled)
y_pred_bgmm = bgmm_final.predict(X_test_scaled)
cm_bgmm = aligned_confusion(y_test.values, y_pred_bgmm)[0]

print(f"Bayesian GMM test ARI: {adjusted_rand_score(y_test, y_pred_bgmm):.3f}")
print(f"Bayesian GMM accuracy: {(cm_bgmm[0,0]+cm_bgmm[1,1])/cm_bgmm.sum():.3f}")